In [ ]:
# Dipendenze gestite via requirements.txt
# pip install -r requirements.txt

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Aggiunge la root del repo al path (per importare config/)
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# Mount Drive se siamo su Colab
def is_colab():
    return "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")

if is_colab():
    from google.colab import drive
    drive.mount('/content/drive')

from config.paths import get_base_path, is_colab


# carica file

In [ ]:
# ============================================================
# CONFIGURAZIONE PARTITA — modifica solo questa sezione
# ============================================================
SEASON          = "2025-2026"
compare_both_teams = True
write_files     = True

# Nomi delle squadre
sq1 = "Decimo"
sq2 = "JVC"

# Path dei file Excel relativi alla cartella base della stagione
# (copialo da config/matches.csv)
filename1 = "(POR) jvc - decimo 3-1+1/[decimo] jvc-decimo.xlsx"
filename2 = "(POR) jvc - decimo 3-1+1/[jvc] jvc-decimo.xlsx"

# ============================================================
# Costruzione path assoluti
# ============================================================
from config.paths import build_base_path

base = build_base_path(season=SEASON)
file_path1 = str(base / filename1)
file_path2 = str(base / filename2)

# Caricamento DataFrame
try:
    if compare_both_teams:
        df1 = pd.read_excel(file_path1, skiprows=1)
        df2 = pd.read_excel(file_path2, skiprows=1)
    else:
        df = pd.read_excel(file_path1, skiprows=1)
except FileNotFoundError as e:
    print(f"File non trovato: {e}")
except Exception as e:
    print(f"Errore: {e}")

if compare_both_teams:
    df1.dropna(subset=['Numero'], inplace=True)
    df1['Numero'] = df1['Numero'].astype(int)
    df1['Giocatore'] = df1['Cognome'] + ' ' + df1['Numero'].astype(str)
    display(df1.head())

    df2.dropna(subset=['Numero'], inplace=True)
    df2['Numero'] = df2['Numero'].astype(int)
    df2['Giocatore'] = df2['Cognome'] + ' ' + df2['Numero'].astype(str)
    display(df2.head())
else:
    df.dropna(subset=['Numero'], inplace=True)
    df['Numero'] = df['Numero'].astype(int)
    df['Giocatore'] = df['Cognome'] + ' ' + df['Numero'].astype(str)
    display(df.head())


In [33]:
#df2.tail(20)


#tabellino formattato

In [34]:
srv_pos = ["#", "+", "/"]
srv_neg = ["="]

In [ ]:
# Funzioni core di efficienza/tabellino estratte in src/efficiency.py (riuso cross-notebook).
from src.efficiency import (
    calcola_efficienza,
    find_errors,
    separate_attacks_counterattacks,
    separate_free_ball,
    calcola_efficienza_free_ball,
    export_tabellino_to_xlsx,
    eff_scalar as _eff_scalar,
    eff_from_calcola as _eff_from_calcola,
)


In [36]:
if compare_both_teams:
  export_tabellino_to_xlsx(
      df=df1,
      filepath=str(base) + "/" + filename1.replace('.xlsx', ' [tabellino F].xlsx'),
      sheet_name="Tabellino",
      top_start_row=0,      # intestazione in riga 1
      bottom_start_row=15   # blocco basso da riga 16
  )
  export_tabellino_to_xlsx(
      df=df2,
      filepath=str(base) + "/" + filename2.replace('.xlsx', ' [tabellino F].xlsx'),
      sheet_name="Tabellino",
      top_start_row=0,      # intestazione in riga 1
      bottom_start_row=15   # blocco basso da riga 16
  )
else:
  export_tabellino_to_xlsx(
      df=df,
      filepath=str(base) + "/" + filename1.replace('.xlsx', ' [tabellino F].xlsx'),
      sheet_name="Tabellino",
      top_start_row=0,      # intestazione in riga 1
      bottom_start_row=15   # blocco basso da riga 16
  )



# Grafico punti per giocatore

In [ ]:
# Funzioni "punti per giocatore" estratte in src/attacks.py (riuso cross-notebook).
from src.attacks import compute_points_table, plot_points_grouped


In [ ]:
# Build the summary table from the existing `df` and plot
if compare_both_teams:
  pts1 = compute_points_table(df1, prefer_surname=False)
  pts2 = compute_points_table(df2, prefer_surname=False)
  save_dir1 = "/".join(file_path1.split('/')[:-1])
  save_dir2 = "/".join(file_path2.split('/')[:-1])
  ax1 = plot_points_grouped(pts1, sq1, title="Punti per giocatore — Muro, Battuta, Attacco, Totale", save_dir=save_dir1, write_files=write_files)
  ax2 = plot_points_grouped(pts2, sq2, title="Punti per giocatore — Muro, Battuta, Attacco, Totale", save_dir=save_dir2, write_files=write_files)
  display(pts1.reset_index())
  display(pts2.reset_index())
else:
  pts = compute_points_table(df, prefer_surname=False)  # assumes df has columns "Tipo" and "Voto"
  save_dir1 = "/".join(file_path1.split('/')[:-1])
  ax = plot_points_grouped(pts, sq1, title="Punti per giocatore — Muro, Battuta, Attacco, Totale", save_dir=save_dir1, write_files=write_files)
  display(pts.reset_index())


# Grafico efficienza attacco

In [ ]:
# Funzioni "efficienza attacco" estratte in src/attacks.py (riuso cross-notebook).
from src.attacks import compute_attack_eff_breakdown, plot_attack_eff_breakdown_bars, create_attack_eff_plots


In [ ]:
# Attack efficiencies: Total • After reception • Counterattack
if compare_both_teams:
  create_attack_eff_plots(df1, sq=sq1, save_dir="/".join(file_path1.split('/')[:-1]), write_files=write_files)
  create_attack_eff_plots(df2, sq=sq2, save_dir="/".join(file_path2.split('/')[:-1]), write_files=write_files)
else:
  create_attack_eff_plots(df, sq=sq1, save_dir="/".join(file_path1.split('/')[:-1]), write_files=write_files)


# trend per set

In [ ]:
# Funzioni "trend per set" estratte in src/attacks.py (riuso cross-notebook).
from src.attacks import plot_set_efficiency_groups, create_metrics_plot


In [ ]:
if compare_both_teams:
  create_metrics_plot(df1, sq=sq1, save_dir="/".join(file_path1.split('/')[:-1]), write_files=write_files)
  create_metrics_plot(df2, sq=sq2, save_dir="/".join(file_path2.split('/')[:-1]), write_files=write_files)
else:
  create_metrics_plot(df, sq=sq1, save_dir="/".join(file_path1.split('/')[:-1]), write_files=write_files)


# radar

In [ ]:
# plot_set_radar estratta in src/attacks.py (riuso cross-notebook).
from src.attacks import plot_set_radar


In [ ]:
#sets=[4]
all_sets = df1['Numero Set'].unique()
for sets in all_sets:
  sets = [sets]
  if compare_both_teams:
    plot_set_radar(
        df_raw=df1,
        df_raw_b=df2,
        sets=sets,
        labels=(sq1,sq2),
        rmin=-100, rmax=100, show_errors="annotate",
        save_dir="/".join(file_path1.split('/')[:-1]), write_files=write_files,
        )
  else:
    plot_set_radar(
        df_raw=df,
        sets=sets,
        labels=(sq1,),
        rmin=-100, rmax=100, show_errors="annotate",
        save_dir="/".join(file_path1.split('/')[:-1]), write_files=write_files,
        )


In [45]:
#df2[(df2['Tipo']=='attacco') & (df2['Voto']=='=')].groupby('Numero Set').size()
int(sets[0])

5

# altro

In [31]:
licenziati = pd.DataFrame()
licenziati = pd.concat([licenziati, df2[(df2['Numero Set']==1) & (df2['Punti Locali']<=22) & ((df2['Punti Ospiti']<=15))]])
licenziati = pd.concat([licenziati, df2[(df2['Numero Set']==3) & (df2['Punti Locali']>=11) & ((df2['Punti Ospiti']>=16))]])
licenziati = pd.concat([licenziati, df2[(df2['Numero Set']==4) & (df2['Punti Locali']<=20) & ((df2['Punti Ospiti']<=15))]])

caranzetti = df2[~df2['id'].isin(licenziati['id'].values)]

In [ ]:
caranzetti.id.values

array([ 88,  89,  90,  91,  92,  93,  94,  95,  96,  97,  98,  99, 100,
       101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113,
       114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126,
       127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139,
       140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152,
       153, 154, 155, 156, 158, 159, 160, 161, 162, 163, 164, 165, 166,
       167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179,
       180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192,
       193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205,
       206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218,
       219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231,
       232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244,
       245, 246, 247, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258,
       259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 27

In [ ]:
export_tabellino_to_xlsx(
      df=licenziati,
      filepath=str(base) + "/" + filename1.replace('.xlsx', ' [tab Licenziati].xlsx'),
      sheet_name="Tabellino",
      top_start_row=0,      # intestazione in riga 1
      bottom_start_row=15   # blocco basso da riga 16
  )
export_tabellino_to_xlsx(
      df=caranzetti,
      filepath=str(base) + "/" + filename1.replace('.xlsx', ' [tab Caranzetti].xlsx'),
      sheet_name="Tabellino",
      top_start_row=0,      # intestazione in riga 1
      bottom_start_row=15   # blocco basso da riga 16
  )

'/content/drive/MyDrive/Pallavolo/Decimo Roma/2025-2026/Serie D/Match analysis/(a5) Roma7 - Decimo 2-3/[Roma7] Roma7 - decimo [tab Caranzetti].xlsx'

# tabellino

In [ ]:
def _attack_tipo_value(df, tipo_col="Tipo"):
    vals = pd.Series(df[tipo_col].dropna().astype(str).unique())
    cand = vals[vals.str.contains("att", case=False, na=False)]
    exact = cand[cand == "Attacco"]
    return exact.iloc[0] if not exact.empty else (cand.iloc[0] if not cand.empty else "Attacco")

def _eff_scalar(res):
    """Estrae lo scalare dalla tua calcola_efficienza(total_efficiency=True)."""
    if isinstance(res, pd.DataFrame) and "Efficienza Totale" in res.columns:
        v = res["Efficienza Totale"].iloc[0]
    else:
        v = float(res)
    return (v*100) if -1.0 <= v <= 1.0 else v

def _eff_from_calcola(df_subset, tipo_val, pos, neg):
    """Wrapper pratico per ottenere direttamente la % numerica."""
    return _eff_scalar(calcola_efficienza(
        df=df_subset, tipo=tipo_val, pos=list(pos), neg=list(neg), total_efficiency=True
    ))



def calcola_efficienza(df, tipo, pos=['#', '+'], neg=['-', '='],
                       total_efficiency=False, min_val=0, set=all):
    # Filter the DataFrame for the given 'tipo'
    filtered_df = df[df['Tipo'] == tipo]
    if set != all:
      filtered_df = filtered_df[filtered_df['Numero Set'] == set]

    if total_efficiency:
        positive_rows = filtered_df[filtered_df['Voto'].isin(pos)].shape[0]
        negative_rows = filtered_df[filtered_df['Voto'].isin(neg)].shape[0]
        total_rows = filtered_df.shape[0]

        if total_rows > 0:
            efficiency = (positive_rows - negative_rows) / total_rows
        else:
            efficiency = 0

        return pd.DataFrame([{'Tipo': tipo, 'Efficienza Totale': efficiency, 'Effettuati': total_rows}])

    else:
        grouped = filtered_df.groupby('Giocatore')

        results = []
        for player, player_df in grouped:
            positive_rows = player_df[player_df['Voto'].isin(pos)].shape[0]
            negative_rows = player_df[player_df['Voto'].isin(neg)].shape[0]
            total_rows = player_df.shape[0]

            if total_rows >= min_val:
                efficiency = (positive_rows - negative_rows) / total_rows
                player_result = {'Giocatore': player, 'Eff': efficiency, 'Tot': total_rows}

                # Add columns for all existing vote types in player_df
                for voto in player_df['Voto'].unique():
                    player_result[voto] = player_df[player_df['Voto'] == voto].shape[0]

                results.append(player_result)

        results_df = pd.DataFrame(results)

        # Calculate total efficiency and total counts for each vote type
        total_positive = filtered_df[filtered_df['Voto'].isin(pos)].shape[0]
        total_negative = filtered_df[filtered_df['Voto'].isin(neg)].shape[0]
        total_actions = filtered_df.shape[0]

        if total_actions > 0:
            overall_efficiency = (total_positive - total_negative) / total_actions
        else:
            overall_efficiency = 0

        total_row = {'Giocatore': 'Totale', 'Eff': overall_efficiency, 'Tot': total_actions}
        for voto in filtered_df['Voto'].unique():
             total_row[voto] = filtered_df[filtered_df['Voto'] == voto].shape[0]

        # Append the total row to the results DataFrame
        total_row_df = pd.DataFrame([total_row])
        results_df = pd.concat([results_df, total_row_df], ignore_index=True)

        return results_df

def find_errors(df):
    """
    Finds all errors (Voto = '=') for 'battuta', 'attacco', and 'alzata' types
    and returns a DataFrame with error counts per player and type.

    Args:
        df: The pandas DataFrame containing the data.

    Returns:
        A pandas DataFrame with error counts per player for 'battuta', 'attacco', and 'alzata'.
    """
    error_types = ['battuta', 'attacco', 'alzata']
    errors_df = df[(df['Tipo'].isin(error_types)) & (df['Voto'] == '=')]

    # Group by player and type and count the errors
    error_counts = errors_df.groupby(['Giocatore', 'Tipo']).size().unstack(fill_value=0)

    # Ensure all error types are present as columns, filling with 0 if a player has no errors of that type
    for error_type in error_types:
        if error_type not in error_counts.columns:
            error_counts[error_type] = 0

    # Calculate the total errors for each type
    total_errors = error_counts[error_types].sum()

    # Create a new row for the totals
    total_row = pd.DataFrame(total_errors).T
    total_row.index = ['Totale']

    # Concatenate the original error_counts DataFrame with the total_row
    error_counts_with_total = pd.concat([error_counts[error_types], total_row])
    error_counts_with_total = error_counts_with_total.reset_index().rename(columns={'index':'Giocatore'})

    return error_counts_with_total


def separate_attacks_counterattacks(df, rec_vote=["#","+","!","-"]):
    """
    Ritorna:
      - attacchi dopo ricezione (con o senza alzata intermedia)
      - contrattacchi (tutti gli altri attacchi)
    Usa una copia normalizzata (minuscolo) per il matching, ma restituisce righe dal df originale.
    """
    if "Tipo" not in df.columns or "Voto" not in df.columns:
        raise ValueError("Mancano colonne 'Tipo' o 'Voto' nel DataFrame.")

    d = df.copy()
    d["_tipo_lc"] = d["Tipo"].astype(str).str.lower()
    d["_voto"]    = d["Voto"].astype(str).str.strip()

    idx_after_rec, idx_counter = [], []
    for i in range(len(d)):
        if d.iloc[i]["_tipo_lc"] != "attacco":
            continue

        after_reception = False
        # ... , Ricezione, Attacco
        if i-1 >= 0 and d.iloc[i-1]["_tipo_lc"] == "ricezione" and d.iloc[i-1]["_voto"] in rec_vote:
            after_reception = True
        # ... , Ricezione, Alzata, Attacco
        if not after_reception and i-2 >= 0:
            if (d.iloc[i-2]["_tipo_lc"] == "ricezione" and d.iloc[i-2]["_voto"] in rec_vote
                and d.iloc[i-1]["_tipo_lc"] == "alzata"):
                after_reception = True

        (idx_after_rec if after_reception else idx_counter).append(df.index[i])

    return df.loc[idx_after_rec], df.loc[idx_counter]


def create_player_summary_df(df):
    """
    Creates a DataFrame with a MultiIndex column structure summarizing player statistics.

    Aggiunte/aggiornate:
      - Att(R#+): attacchi successivi a ricezione in {"#","+"}
      - Att(R!):  attacchi successivi a ricezione in {"!"}
      - Att(R-):  attacchi successivi a ricezione in {"-"}
      - Attacco(rice tot): attacchi dopo qualsiasi ricezione ({"#","+", "!", "-"})
      - Contrattacco: tutti gli altri attacchi
    """
    # Fondamentali base
    battuta_eff = calcola_efficienza(df, 'battuta', pos=['#', '+', '/', '!'], neg=['='])
    attacco_eff = calcola_efficienza(df, 'attacco', pos=['#'], neg=['=', '/'])
    muro_eff    = calcola_efficienza(df, 'muro',    pos=['#', '+', ], neg=['=', '/'])
    rice_eff    = calcola_efficienza(df, 'ricezione', pos=['#', '+'], neg=['=', '/'])
    errors      = find_errors(df)

    # Att(R#+): ricezione positiva
    pos_rec_attacks, _  = separate_attacks_counterattacks(df, rec_vote=["#", "+"])
    pos_rec_att_eff     = calcola_efficienza(pos_rec_attacks, 'attacco', pos=['#'], neg=['=', '/'])

    # Att(R-): ricezione negativa
    neg_rec_attacks, _  = separate_attacks_counterattacks(df, rec_vote=["-"])
    neg_rec_att_eff     = calcola_efficienza(neg_rec_attacks, 'attacco', pos=['#'], neg=['=', '/'])

    # Att(R!): ricezione esclamativa
    escl_rec_attacks, _ = separate_attacks_counterattacks(df, rec_vote=["!"])
    escl_rec_att_eff    = calcola_efficienza(escl_rec_attacks, 'attacco', pos=['#'], neg=['=', '/'])

    # Totale ricezione e contrattacchi
    all_rec_attacks, counterattacks = separate_attacks_counterattacks(df)  # default ["#","+","!","-"]
    all_rec_att_eff    = calcola_efficienza(all_rec_attacks, 'attacco', pos=['#'], neg=['=', '/'])
    counterattacks_eff = calcola_efficienza(counterattacks, 'attacco', pos=['#'], neg=['=', '/'])

    # Unione elenco giocatori su tutte le tabelle disponibili
    all_players = set(battuta_eff['Giocatore']).union(
        rice_eff['Giocatore'],
        all_rec_att_eff['Giocatore'],
        pos_rec_att_eff['Giocatore'],
        neg_rec_att_eff['Giocatore'],
        escl_rec_att_eff['Giocatore'],
        counterattacks_eff['Giocatore'],
        muro_eff['Giocatore'],
        errors['Giocatore']
    )

    # MultiIndex: con rinomina delle tre colonne richieste
    sections = [
        'Battuta', 'Ricezione', 'Attacco(rice tot)',
        'Att(R#+)', 'Att(R!)', 'Att(R-)',
        'Contrattacco', 'Muro', 'Errori'
    ]
    multiindex_cols = pd.MultiIndex.from_product([sections, []])

    # DataFrame vuoto
    player_summary_df = pd.DataFrame(index=sorted(list(all_players)), columns=multiindex_cols, dtype=object)

    # Helper per popolare (aggiunge dinamicamente le sotto-colonne)
    def populate_stats(df_source, col_level1, df_dest):
        if df_source is not None and not df_source.empty: # check per df vuoto
            for _, row in df_source.iterrows():
                player = row['Giocatore']
                existing_level2 = df_dest.columns.get_level_values(level=1)[
                    df_dest.columns.get_level_values(level=0) == col_level1
                ].tolist()
                new_level2 = [col for col in row.drop('Giocatore').index.tolist() if col not in existing_level2]
                if new_level2:
                    new_cols = pd.MultiIndex.from_product([[col_level1], new_level2])
                    df_dest = df_dest.reindex(columns=df_dest.columns.tolist() + new_cols.tolist())
                for col in row.drop('Giocatore').index:
                    df_dest.loc[player, (col_level1, col)] = row[col]
        return df_dest

    # Popola tutte le sezioni
    player_summary_df = populate_stats(battuta_eff,          'Battuta',            player_summary_df)
    player_summary_df = populate_stats(rice_eff,             'Ricezione',          player_summary_df)
    player_summary_df = populate_stats(all_rec_att_eff,      'Attacco(rice tot)',  player_summary_df)
    player_summary_df = populate_stats(pos_rec_att_eff,      'Att(R#+)',           player_summary_df)  # rinominata
    player_summary_df = populate_stats(escl_rec_att_eff,     'Att(R!)',            player_summary_df)  # nuova + rinominata
    player_summary_df = populate_stats(neg_rec_att_eff,      'Att(R-)',            player_summary_df)  # rinominata
    player_summary_df = populate_stats(counterattacks_eff,   'Contrattacco',       player_summary_df)
    player_summary_df = populate_stats(muro_eff,             'Muro',               player_summary_df)
    player_summary_df = populate_stats(errors,               'Errori',             player_summary_df)

    # Formattazione: percentuali su 'Eff'/'Pos', conteggi come int, vuoti -> '-'
    for col_level1 in sections:
        if col_level1 in player_summary_df.columns.get_level_values(level=0):
            for col_level2 in player_summary_df[col_level1].columns:
                if col_level2 in ['Eff', 'Pos']:
                    numeric_col = pd.to_numeric(player_summary_df[(col_level1, col_level2)], errors='coerce')
                    player_summary_df[(col_level1, col_level2)] = numeric_col.apply(
                        lambda x: '{:.0%}'.format(x) if pd.notna(x) else '-'
                    )
                else:
                    player_summary_df[(col_level1, col_level2)] = (
                        pd.to_numeric(player_summary_df[(col_level1, col_level2)], errors='coerce')
                        .fillna(-1).astype(int).replace(-1, '-')
                    )

    return player_summary_df


In [ ]:
if compare_both_teams:
  player_stats_summary1 = create_player_summary_df(df1)
  player_stats_summary2 = create_player_summary_df(df2)
  display(player_stats_summary1)
  display(player_stats_summary2)
else:
  player_stats_summary = create_player_summary_df(df)
  display(player_stats_summary)

Battuta                           Ricezione      ... Muro      \
                  Eff Tot   =   +  #   -  /   !       Eff Tot  ...  Tot   =   
Caranzetti 17     -8%  13   4   1  1   6  1   -         -   -  ...    6   3   
Carrer 18         33%  15   2   3  -   6  -   4      100%   3  ...   11   4   
Cepparano 11      40%  15   1   2  1   7  3   1       44%  18  ...    5   4   
Chimenton 4       21%  14   3   -  -   5  1   5       38%  21  ...    3   1   
D’Arienzo 22       0%   9   2   2  -   5  -   -         -   -  ...    3   1   
Licenziati  1    100%   1   -   -  -   -  -   1         -   -  ...    -   -   
Moscetta 30       25%  12   2   2  -   5  1   2         -   -  ...    8   2   
Pedico 31           -   -   -   -  -   -  -   -       42%  33  ...    -   -   
Sardella 66       33%   6   -   -  1   4  -   1         -   -  ...    -   -   
Totale            22%  85  14  10  3  38  6  14       44%  75  ...   36  15   

                              Errori                 
               /  !  #  -  + battuta attacco alzata  
Caranzetti 17  1  1  1  -  -       4       0      3  
Carrer 18      -  2  4  1  -       2       1      0  
Cepparano 11   -  -  -  1  -       1       2      0  
Chimenton 4    -  1  1  -  -       3       1      0  
D’Arienzo 22   -  -  1  -  1       2       4      0  
Licenziati  1  -  -  -  -  -       0       0      1  
Moscetta 30    1  2  2  -  1       2       0      2  
Pedico 31      -  -  -  -  -       0       0      1  
Sardella 66    -  -  -  -  -       -       -      -  
Totale         2  6  9  2  2      14       8      7  

[10 rows x 65 columns]

Battuta                           Ricezione      ...  Muro      \
                 Eff Tot  /   =   -   +   !  #       Eff Tot  ...   Eff Tot   
Bova  3            -   -  -   -   -   -   -  -       61%  23  ...  100%   1   
Caridi 9         -9%  11  1   4   4   1   1  -       50%   4  ...   17%  12   
Colella 6        12%   8  -   1   5   2   -  -        0%   2  ...  100%   2   
Frosi 15         36%  14  1   2   5   1   2  3       29%  17  ...  -22%   9   
Leonardi 1       -6%  18  -   4  11   -   2  1         -   -  ...  -33%   3   
Martinoia 13      8%  13  1   3   6   2   1  -         -   -  ...    0%   5   
Messina 31        0%   2  -   1   -   -   1  -       50%   2  ...     -   -   
Pozza 5           0%   1  -   -   1   -   -  -      100%   1  ...  100%   1   
Tocci 16          0%   1  -   -   1   -   -  -         -   -  ...     -   -   
Totale           14%  95  4  20  42  11  13  5       41%  71  ...    8%  39   
Tufi 8             -   -  -   -   -   -   -  -         -   -  ...  100%   1   
Zdragon 14       38%   8  1   1   3   2   1  -       50%   2  ...    0%   3   
Zorli 4          26%  19  -   4   6   3   5  1       25%  20  ...  -50%   2   

                               Errori                 
              +  !   #  -   = battuta attacco alzata  
Bova  3       1  -   -  -   -       -       -      -  
Caridi 9      2  1   4  1   4       4       1      0  
Colella 6     -  -   2  -   -       1       0      0  
Frosi 15      2  3   -  -   4       2       0      0  
Leonardi 1    -  -   1  -   2       4       0      1  
Martinoia 13  -  3   1  -   1       3       3      0  
Messina 31    -  -   -  -   -       1       0      0  
Pozza 5       -  -   1  -   -       -       -      -  
Tocci 16      -  -   -  -   -       -       -      -  
Totale        5  9  11  1  13      20       7      1  
Tufi 8        -  -   1  -   -       -       -      -  
Zdragon 14    -  1   1  -   1       1       0      0  
Zorli 4       -  1   -  -   1       4       3      0  

[13 rows x 61 columns]

In [ ]:
# esegui per scrivere il tabellino
if write_files:
  if compare_both_teams:
    player_stats_summary1.to_excel(str(base) + "/" + filename1.replace('.xlsx', ' [tabellino].xlsx'))
    player_stats_summary2.to_excel(str(base) + "/" + filename2.replace('.xlsx', ' [tabellino].xlsx'))
  else:
    player_stats_summary.to_excel(str(base) + "/" + filename.replace('.xlsx', ' [tabellino].xlsx'))

# SO per rotazioni

In [ ]:
#neg_rec_attacks, _ = separate_attacks_counterattacks(df, rec_vote=["-","!"])
#neg_rec_att_eff = calcola_efficienza(neg_rec_attacks, 'attacco', pos=['#'], neg=['=', '/'])
#pos_rec_attacks, _ = separate_attacks_counterattacks(df, rec_vote=["#","+"])
#pos_rec_att_eff = calcola_efficienza(pos_rec_attacks, 'attacco', pos=['#'], neg=['=', '/'])
#all_rec_attacks, counterattacks = separate_attacks_counterattacks(df)
#all_rec_att_eff = calcola_efficienza(all_rec_attacks, 'attacco', pos=['#'], neg=['=', '/'])

In [ ]:
# Function to apply calcola_efficienza to each group
#def apply_calcola_efficienza(group):
#    return calcola_efficienza(group, tipo='attacco', pos=['#'], neg=['=', '/'], total_efficiency=True)
#
## Group by 'Posizione Palleggiatore' and apply the function
#neg_rec_attacks_by_pos = neg_rec_attacks.groupby('Posizione Palleggiatore').apply(apply_calcola_efficienza, include_groups=False)
#pos_rec_attacks_by_pos = pos_rec_attacks.groupby('Posizione Palleggiatore').apply(apply_calcola_efficienza, include_groups=False)
#all_rec_attacks_by_pos = all_rec_attacks.groupby('Posizione Palleggiatore').apply(apply_calcola_efficienza, include_groups=False)
#
## Display the result
#all_rec_attacks_by_pos

In [ ]:
esc_rec_attacks, _ = separate_attacks_counterattacks(df1, rec_vote=["!"])
esc_rec_attacks

,id,Tipo,Voto,Cognome,Numero,Posizione Giocatore,Posizione Palleggiatore,Numero Set,Punti Locali,Punti Ospiti,Giocatore
45,45,attacco,!,Opposto,99,p2,p6,1,6,9,Opposto 99
82,82,attacco,-,De vincenzo,15,p4,p6,1,12,16,De vincenzo 15
112,112,attacco,+,Caruso,23,p4,p5,1,17,22,Caruso 23
116,116,attacco,+,Virgilio,21,p2,p5,1,18,22,Virgilio 21
152,152,attacco,-,De vincenzo,15,p4,p6,2,4,3,De vincenzo 15
203,203,attacco,-,Caruso,23,p2,p1,2,17,12,Caruso 23
208,208,attacco,-,Opposto,99,p4,p1,2,19,12,Opposto 99
256,256,attacco,-,Cavallaro,88,p4,p5,3,7,5,Cavallaro 88
262,262,attacco,/,Cavallaro,88,p4,p4,3,8,6,Cavallaro 88
264,264,attacco,#,Cavallaro,88,p4,p4,3,9,6,Cavallaro 88


from matplotlib import pyplot as plt
esc_rec_attacks['id'].plot(kind='hist', bins=20, title='id')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
esc_rec_attacks['Numero'].plot(kind='hist', bins=20, title='Numero')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
esc_rec_attacks['Numero Set'].plot(kind='hist', bins=20, title='Numero Set')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
esc_rec_attacks['Punti Locali'].plot(kind='hist', bins=20, title='Punti Locali')
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(esc_rec_attacks['Cognome'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(esc_rec_attacks, x='id', y='Cognome', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(esc_rec_attacks['Posizione Giocatore'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(esc_rec_attacks, x='id', y='Posizione Giocatore', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(esc_rec_attacks['Posizione Palleggiatore'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(esc_rec_attacks, x='id', y='Posizione Palleggiatore', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)